# 02 — Modelos: RF, SVM, MLP
Entrenamiento y evaluación de los 3 modelos.

> **Orden recomendado:** ejecutar primero `01_eda_limpieza.ipynb` y luego `03_experimento.ipynb`.  
> Este notebook carga las **mejores hiperparámetros** desde `results/best_configs.json` (generado por el experimento).  
> Si el archivo no existe, usa valores por defecto y muestra una advertencia.

In [1]:
import sys
sys.path.append('../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedShuffleSplit

import json
import os

from preprocessing import encode_and_split, TARGET_COL
from evaluation import evaluate
from plots import plot_confusion_matrix, plot_confusion_matrices_combined, plot_metrics_bar, plot_feature_importance, plot_mlp_loss
from paths import ensure_results_dirs
from best_models import load_best_configs, SVM_SAMPLE

PROCESSED_PATH = '../data/processed/datos_limpios.csv'
CM_PATH        = '../results/confusion_matrices/'
RESULTS_PATH   = '../results/'
BEST_CFG_PATH  = '../results/best_configs.json'

ensure_results_dirs('../results')

# Cargar mejores hiperparámetros del experimento (notebook 03)
if os.path.isfile(BEST_CFG_PATH):
    BEST = load_best_configs(BEST_CFG_PATH)
    print('✅ Hiperparámetros cargados desde results/best_configs.json')
else:
    print('⚠️ Ejecuta primero notebooks/03_experimento.ipynb (o scripts/run_experiment.py)')
    BEST = {
        'RF': {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 5},
        'SVM': {'C': 10, 'kernel': 'rbf', 'gamma': 'scale'},
        'MLP': {'hidden_layer_sizes': (100, 50), 'activation': 'relu', 'learning_rate_init': 0.001},
    }

sns.set_theme(style='whitegrid')
print('Librerías cargadas')

Librerías cargadas


## 1. Carga y split de datos

In [2]:
# Leemos el CSV ya limpio generado por el notebook 01.
# encode_and_split hace todo de una:
#   1. LabelEncoder al target (0=Ideación, 1=Intento)
#   2. LabelEncoder a variables categóricas (sexo, localidad, etc.)
#   3. SimpleImputer con mediana para NaN restantes
#   4. Train/test split 80/20 estratificado (mantiene proporción 73/27)
#   5. StandardScaler — necesario para SVM y MLP (sensibles a escala)
# Devuelve versión sin escalar (X_train) para RF, y escalada (X_train_sc) para SVM/MLP
df = pd.read_csv(PROCESSED_PATH)
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
print(f'Balance de clases: {df[TARGET_COL].value_counts().to_dict()}')

X_train, X_test, X_train_sc, X_test_sc, y_train, y_test, le = encode_and_split(df)
print(f'\nTrain: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Clases: {le.classes_}  →  0={le.classes_[0]}, 1={le.classes_[1]}')

Filas: 254,251 | Columnas: 16
Balance de clases: {'Ideación suicida': 184462, 'Intento de Suicidio': 69789}



Train: 203,400 | Test: 50,851
Clases: ['Ideación suicida' 'Intento de Suicidio']  →  0=Ideación suicida, 1=Intento de Suicidio


## 2. Random Forest

In [3]:
# Random Forest: ensemble de árboles de decisión entrenados en subconjuntos aleatorios.
# Ventajas: robusto a outliers, maneja bien features mixtas, da importancia de variables.
# Hiperparámetros clave usados:
#   - n_estimators=200: más árboles = más estable (diminishing returns > 300)
#   - max_depth=20: profundidad suficiente para capturar interacciones sin overfitting total
#   - min_samples_split=5: evita que los árboles memoricen casos individuales
#   - class_weight='balanced': ajusta pesos por el desbalanceo 73%/27% del dataset
#   - n_jobs=-1: usa todos los núcleos del CPU → entrenamiento paralelo más rápido
rf = RandomForestClassifier(
    n_estimators=BEST['RF']['n_estimators'],
    max_depth=BEST['RF']['max_depth'],
    min_samples_split=BEST['RF']['min_samples_split'],
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
print(f"RF config: {BEST['RF']}")
rf.fit(X_train, y_train)

y_pred_rf, metrics_rf, report_rf = evaluate(rf, X_test, y_test, le)
print('=== Random Forest ===')
print(report_rf)
print('Métricas:', metrics_rf)

=== Random Forest ===
                     precision    recall  f1-score   support

   Ideación suicida       0.81      0.78      0.79     36893
Intento de Suicidio       0.47      0.52      0.49     13958

           accuracy                           0.71     50851
          macro avg       0.64      0.65      0.64     50851
       weighted avg       0.72      0.71      0.71     50851

Métricas: {'accuracy': 0.7065, 'f1': 0.711, 'precision': 0.7169, 'recall': 0.7065}


In [4]:
plot_confusion_matrix(y_test, y_pred_rf, le.classes_, 'Matriz de Confusión — Random Forest', CM_PATH + 'cm_random_forest.png')
plot_feature_importance(rf, X_train.columns, RESULTS_PATH + 'eda/feature_importance_rf.png')

## 3. SVM
> **Nota técnica:** SVM con kernel RBF tiene complejidad O(n²) en memoria y O(n³) en tiempo.  
> Con 254,000 filas el entrenamiento sería inviable (días). Se usa una muestra estratificada  
> de 15,000 registros — práctica estándar y estadísticamente representativa con este tamaño de dataset.

In [5]:
# StratifiedShuffleSplit garantiza que la muestra de 15k mantiene la misma proporción
# de clases que el dataset completo (73% ideación / 27% intento).
# Usamos índices posicionales (numpy) porque X_train_sc es array, no DataFrame.
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, train_size=SVM_SAMPLE, random_state=42)
svm_idx, _ = next(sss.split(X_train_sc, y_train))

X_train_svm = X_train_sc[svm_idx]
y_train_svm = y_train.iloc[svm_idx]

# SVM: encuentra el hiperplano óptimo que maximiza el margen entre clases.
# Kernel RBF proyecta los datos a un espacio de mayor dimensión → captura relaciones no lineales.
#   - C=10: penaliza fuerte los errores de clasificación (margen más estrecho pero más preciso)
#   - gamma='scale': ajusta automáticamente según la varianza de los datos
#   - class_weight='balanced': compensa el desbalanceo 73/27
svm = SVC(
    C=BEST['SVM']['C'],
    kernel=BEST['SVM']['kernel'],
    gamma=BEST['SVM']['gamma'],
    class_weight='balanced',
    random_state=42
)
print(f"SVM config (experimento): {BEST['SVM']}")
svm.fit(X_train_svm, y_train_svm)

# Evaluamos sobre el test completo (50,850 filas) para métricas comparables con RF y MLP
y_pred_svm, metrics_svm, report_svm = evaluate(svm, X_test_sc, y_test, le)
print('=== SVM ===')
print(report_svm)
print('Métricas:', metrics_svm)

=== SVM ===
                     precision    recall  f1-score   support

   Ideación suicida       0.81      0.68      0.74     36893
Intento de Suicidio       0.40      0.57      0.47     13958

           accuracy                           0.65     50851
          macro avg       0.61      0.63      0.61     50851
       weighted avg       0.70      0.65      0.67     50851

Métricas: {'accuracy': 0.6497, 'f1': 0.6651, 'precision': 0.697, 'recall': 0.6497}


In [6]:
plot_confusion_matrix(y_test, y_pred_svm, le.classes_, 'Matriz de Confusión — SVM', CM_PATH + 'cm_svm.png')

## 4. MLP (Red Neuronal)

In [7]:
# MLP (Red Neuronal Multicapa): aprende representaciones jerárquicas de los datos
# a través de capas de neuronas con funciones de activación no lineales.
# Hiperparámetros clave:
#   - hidden_layer_sizes=(100, 50): 2 capas ocultas con 100 y 50 neuronas
#   - activation='relu': función de activación recomendada para datos tabulares
#   - learning_rate_init=0.001: tasa de aprendizaje inicial del optimizador Adam
#   - early_stopping=True: detiene el entrenamiento si la validación no mejora
#     → evita overfitting sin necesidad de fijar max_iter manualmente
#   - validation_fraction=0.1: 10% del train se usa para monitorear la convergencia
mlp = MLPClassifier(
    hidden_layer_sizes=BEST['MLP']['hidden_layer_sizes'],
    activation=BEST['MLP']['activation'],
    learning_rate_init=BEST['MLP']['learning_rate_init'],
    max_iter=200,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    verbose=False
)
print(f"MLP config (experimento): {BEST['MLP']}")
mlp.fit(X_train_sc, y_train)

y_pred_mlp, metrics_mlp, report_mlp = evaluate(mlp, X_test_sc, y_test, le)
print('=== MLP ===')
print(f'Épocas hasta convergencia: {mlp.n_iter_}')
print(report_mlp)
print('Métricas:', metrics_mlp)

=== MLP ===
Épocas hasta convergencia: 23
                     precision    recall  f1-score   support

   Ideación suicida       0.76      0.96      0.85     36893
Intento de Suicidio       0.69      0.21      0.33     13958

           accuracy                           0.76     50851
          macro avg       0.73      0.59      0.59     50851
       weighted avg       0.74      0.76      0.71     50851

Métricas: {'accuracy': 0.7577, 'f1': 0.7081, 'precision': 0.7432, 'recall': 0.7577}


In [8]:
plot_mlp_loss(mlp, RESULTS_PATH + 'training/mlp_loss_curve.png')

# Matrices de confusión individuales (para guardar por separado)
plot_confusion_matrix(y_test, y_pred_rf,  le.classes_, 'Matriz de Confusión — Random Forest', CM_PATH + 'cm_random_forest.png')
plot_confusion_matrix(y_test, y_pred_svm, le.classes_, 'Matriz de Confusión — SVM',           CM_PATH + 'cm_svm.png')
plot_confusion_matrix(y_test, y_pred_mlp, le.classes_, 'Matriz de Confusión — MLP',           CM_PATH + 'cm_mlp.png')

# Figura combinada para el PPT
plot_confusion_matrices_combined(
    {'Random Forest': y_pred_rf, 'SVM': y_pred_svm, 'MLP': y_pred_mlp},
    y_test, le.classes_,
    CM_PATH + 'cm_combinadas_mejores.png'
)
print('✅ Matrices de confusión guardadas')

✅ Matrices de confusión guardadas


## 5. Comparación de los 3 modelos

In [9]:
# Tabla y gráfica comparativa de los 3 modelos con sus métricas finales.
# Usamos F1-Score weighted como métrica principal porque el dataset está desbalanceado
# (accuracy puede ser engañoso con 73/27 — un modelo que siempre predice "Ideación" 
# tendría 73% de accuracy pero sería inútil clínicamente).
all_metrics = {
    'Random Forest': metrics_rf,
    'SVM':           metrics_svm,
    'MLP':           metrics_mlp
}

df_metrics = pd.DataFrame(all_metrics).T
print(df_metrics.to_string())

# Barplot con valores anotados para visualización clara en el PPT
plot_metrics_bar(all_metrics, RESULTS_PATH + 'comparacion_modelos_experimento.png')

               accuracy      f1  precision  recall
Random Forest    0.7065  0.7110     0.7169  0.7065
SVM              0.6497  0.6651     0.6970  0.6497
MLP              0.7577  0.7081     0.7432  0.7577
